In [1]:
from dataclasses import dataclass
import pandas as pd
from stock_prediction.constants import *
from stock_prediction.utils.common import *
from datetime import datetime
import os
# os.chdir("../")

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    raw_data_file: Path
    test_size: float
    date_column: str
    date_column: str


In [54]:

config = read_yaml(CONFIG_FILE_PATH)
df = pd.read_csv(config.data_ingestion.raw_data_file, parse_dates=["Date"])
df.head()

2026-08-11 11:01:54,202 | INFO | common| YAML file: config\config.yaml loaded successfully.


,Date,Close,High,Low,Open,Volume
0,2000-01-03,0.837724,0.841934,0.761015,0.784870,535796800
1,2000-01-04,0.767096,0.827901,0.757273,0.810128,512377600
2,2000-01-05,0.778321,0.827434,0.770837,0.776450,778321600
3,2000-01-06,0.710966,0.800772,0.710966,0.794225,767972800
4,2000-01-07,0.744644,0.755870,0.714709,0.722192,460734400


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6685 entries, 0 to 6684
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    6685 non-null   datetime64[ns]
 1   Close   6685 non-null   float64       
 2   High    6685 non-null   float64       
 3   Low     6685 non-null   float64       
 4   Open    6685 non-null   float64       
 5   Volume  6685 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 313.5 KB


In [56]:
df.columns = df.columns.str.lower()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6685 entries, 0 to 6684
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    6685 non-null   datetime64[ns]
 1   close   6685 non-null   float64       
 2   high    6685 non-null   float64       
 3   low     6685 non-null   float64       
 4   open    6685 non-null   float64       
 5   volume  6685 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 313.5 KB


In [57]:
df = df.set_index("date")
df.head()

,close,high,low,open,volume
date,,,,,
2000-01-03,0.837724,0.841934,0.761015,0.784870,535796800
2000-01-04,0.767096,0.827901,0.757273,0.810128,512377600
2000-01-05,0.778321,0.827434,0.770837,0.776450,778321600
2000-01-06,0.710966,0.800772,0.710966,0.794225,767972800
2000-01-07,0.744644,0.755870,0.714709,0.722192,460734400


In [68]:
train_size = int(len(df) * 0.8)
train_data = df.close[:train_size]
len(train_data)

5348

In [70]:
len(df.close[train_size:]) + len(train_data) == len(df.close)

True

In [62]:
train_size

5348.0

In [46]:
df.index.is_monotonic_increasing

True

In [37]:
from dataclasses import dataclass

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    raw_data_file: Path
    test_size: float
    target_column: str


In [38]:
from stock_prediction.utils.common import *
from stock_prediction.constants import *
# from stock_prediction.entity.config_entity import DataTransformationConfig

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    def get_data_transformation_config(self):
        config = self.config.data_transformation
        create_directories([config.root_dir])
        data_transformation_config =  DataTransformationConfig(
            root_dir=config.root_dir,
            raw_data_file=config.raw_data_file,
            test_size=config.test_size,
            target_column=config.target_column
        )
        return data_transformation_config

In [86]:
class DataTransformation:
    def __init__(self, config:DataTransformationConfig):
        self.config = config
    def split_data(self):
        data = pd.read_csv(self.config.raw_data_file)
        data = data[["Close"]]
        train_size = int(len(data) * 0.8)
        train_data = data[:train_size]
        test_data = data[train_size:]
        
        assert train_data.index.max() < test_data.index.min(), \
            "Data leakage detected, train/test split overlap"
        logger.info(f"Train date range:{train_data.index.min()} to {train_data.index.max()}")
        logger.info(f"Test date range: {test_data.index.min()} to {test_data.index.max()}")
        train_data.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=True)
        test_data.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=True)

        return train_data, test_data
        
        
        

In [87]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(data_transformation_config)
data_transformation.split_data()

2026-08-11 11:17:04,836 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-11 11:17:04,836 | INFO | common| YAML file: schema.yaml loaded successfully.
2026-08-11 11:17:04,836 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-11 11:17:04,845 | INFO | common| Directory created at: artifacts
2026-08-11 11:17:04,847 | INFO | common| Directory created at: artifacts/data_transformation
2026-08-11 11:17:04,864 | INFO | 970397553| Train date range:0 to 5347
2026-08-11 11:17:04,866 | INFO | 970397553| Test date range: 5348 to 6684


(           Close
 0       0.837724
 1       0.767096
 2       0.778321
 3       0.710966
 4       0.744644
 ...          ...
 5343  116.674591
 5344  118.864044
 5345  119.691177
 5346  122.513161
 5347  122.814827
 
 [5348 rows x 1 columns],
            Close
 5348  124.459366
 5349  126.853188
 5350  129.422180
 5351  127.709534
 5352  130.813675
 ...          ...
 6680  340.079987
 6681  338.190002
 6682  333.429993
 6683  308.910004
 6684  303.420013
 
 [1337 rows x 1 columns])

In [ ]:

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def chronological_split(self, df: pd.DataFrame):
        date_col = self.config.date_column
        test_size = self.config.test_size

        df = df.sort_values(date_col).reset_index(drop=True)
        split_idx = int(len(df) * (1 - test_size))
        train = df.iloc[:split_idx].reset_index(drop=True)
        test = df.iloc[split_idx:].reset_index(drop=True)

        assert train[date_col].max() < test[date_col].min(), \
            "Data leakage detected: train/test date ranges overlap"

        return train, test

    def run(self):
        df = pd.read_csv(self.config.data_path, parse_dates=[self.config.date_column])
        train, test = self.chronological_split(df)

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info(f"Train shape: {train.shape}, Test shape: {test.shape}")
        logger.info(
            f"Train range: {train[self.config.date_column].min()} to "
            f"{train[self.config.date_column].max()}"
        )
        logger.info(
            f"Test range: {test[self.config.date_column].min()} to "
            f"{test[self.config.date_column].max()}"
        )